# Trabalho Prático 5 - Coloração de Grafos com DSatur
**Disciplina:** Resolução de Problemas com Grafos  
**Orientador:** Prof. Me Ricardo Carubbi  

Este notebook contém a implementação do algoritmo **DSatur** para coloração de grafos, aplicado ao mapa político do Brasil. A implementação utiliza as classes de grafo fornecidas pela disciplina e segue os requisitos do arquivo `T5.md`.

## 1. Classes de Suporte (Base da Disciplina)
Abaixo estão as classes `Node`, `LinkIterator`, `Bag` e `Graph`, adaptadas da base `algs4-py`.

In [ ]:
class Node:
    def __init__(self, item, next_node):
        self.item = item
        self.next = next_node

class LinkIterator:
    def __init__(self, current):
        self.current = current

    def __next__(self):
        if self.current is None:
            raise StopIteration()
        else:
            item = self.current.item
            self.current = self.current.next
            return item

class Bag:
    def __init__(self):
        self.first = None
        self.n = 0

    def __str__(self):
        return " ".join(str(i) for i in self)

    def __iter__(self):
        return LinkIterator(self.first)

    def size(self):
        return self.n

    def is_empty(self):
        return self.first is None

    def add(self, item):
        oldfirst = self.first
        self.first = Node(item, oldfirst)
        self.n += 1

class Graph:
    def __init__(self, v):
        self.V = v
        self.E = 0
        self.adj = [Bag() for _ in range(self.V)]

    def __str__(self):
        lines = [f"{self.V} vertices, {self.E} edges"]
        for v in range(self.V):
            neighbors = " ".join(str(w) for w in self.adj[v])
            lines.append(f"{v}: {neighbors}")
        return "\n".join(lines)

    def add_edge(self, v, w):
        v, w = int(v), int(w)
        self.adj[v].add(w)
        self.adj[w].add(v)
        self.E += 1

    def degree(self, v):
        return self.adj[v].size()

    def max_degree(self):
        max_deg = 0
        for v in range(self.V):
            max_deg = max(max_deg, self.degree(v))
        return max_deg

## 2. Implementação do Algoritmo DSatur
O algoritmo DSatur (Degree of Saturation) prioriza a coloração de vértices com maior número de cores distintas em sua vizinhança.

In [ ]:
def dsatur(graph):
    n = graph.V
    colors = [-1] * n
    order = []
    
    # 1. Colorir o vértice de maior grau primeiro
    max_deg = -1
    start_v = -1
    for v in range(n):
        deg = graph.degree(v)
        if deg > max_deg:
            max_deg = deg
            start_v = v
            
    colors[start_v] = 0
    order.append(start_v)
    
    remaining = set(range(n))
    remaining.remove(start_v)
    
    while remaining:
        # 2. Escolher v' que maximiza DS(v')
        max_sat = -1
        candidates = []
        
        for v in remaining:
            neighbor_colors = {colors[neighbor] for neighbor in graph.adj[v] if colors[neighbor] != -1}
            sat_deg = len(neighbor_colors)
            
            if sat_deg > max_sat:
                max_sat = sat_deg
                candidates = [v]
            elif sat_deg == max_sat:
                candidates.append(v)
        
        # Desempate: maior grau no grafo original
        best_v = -1
        max_deg = -1
        for v in candidates:
            deg = graph.degree(v)
            if deg > max_deg:
                max_deg = deg
                best_v = v
        
        # 3. Encontrar a menor cor k viável
        neighbor_colors = {colors[neighbor] for neighbor in graph.adj[best_v] if colors[neighbor] != -1}
        
        color = 0
        while color in neighbor_colors:
            color += 1
            
        colors[best_v] = color
        order.append(best_v)
        remaining.remove(best_v)
        
    return colors, order

def validate_coloring(graph, colors):
    for v in range(graph.V):
        if colors[v] == -1: return False, f"Vértice {v} não colorido."
        for neighbor in graph.adj[v]:
            if colors[v] == colors[neighbor]:
                return False, f"Conflito entre {v} e {neighbor}."
    return True, "Coloração válida."

## 3. Execução Principal
O código abaixo carrega os dados de `brasil.txt`, executa o DSatur e exibe os resultados conforme exigido.

In [ ]:
# Dados de entrada (conforme brasil.txt)
data = """27
50
0 2
0 20
1 4
1 15
1 24
2 12
2 13
2 20
2 21
3 13
4 7
4 8
4 10
4 15
4 16
4 24
4 26
5 14
5 15
5 16
5 19
6 8
6 10
7 10
7 18
8 10
8 11
8 12
8 26
9 13
9 16
9 26
10 11
10 18
10 25
11 12
11 17
11 25
12 13
12 20
12 26
13 21
13 26
14 15
14 19
15 16
17 23
17 25
18 25
22 23
"""

lines = data.strip().split('\n')
V = int(lines[0])
E = int(lines[1])
g = Graph(V)
for i in range(2, 2 + E):
    v, w = lines[i].split()
    g.add_edge(v, w)

print("--- Lista de Adjacência ---")
print(g)
print()

colors, order = dsatur(g)

states = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]

print("--- Ordem de Coloração ---")
print(" -> ".join([states[v] for v in order]))
print()

print("--- Cores Atribuídas ---")
for v in range(g.V):
    print(f"{states[v]} ({v}): Cor {colors[v]}")
print()

print(f"Total de cores utilizadas: {len(set(colors))}")
is_valid, msg = validate_coloring(g, colors)
print(f"Validação: {msg}")